# UniDAV — Python quickstart

`unidav` is a Cython extension over the UniDAV C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install unidav
```

UniDAV reads and writes vCard and iCalendar, and speaks CardDAV and CalDAV.
Its rule is lossless: the ordered document tree is the source of truth, and a
property the library does not understand survives a round trip untouched.

CI executes this notebook against the wheel the release actually publishes, so
an output below that stops matching fails the build.

## The API

In [1]:
import json

import unidav

unidav.version()

'0.1.0'

## Validate, then normalise

Validation reports; it does not throw. A document that is wrong is still a
document you can look at, which is what importing a file from an unknown
source needs.

In [2]:
card = ("begin:vcard\nversion:4.0\nuid:urn:uuid:ada\n"
        "fn:Ada Lovelace\nemail:ada@example.org\n"
        "x-phone-model:something\nend:vcard\n")

unidav.validate(card)

{'valid': True, 'kind': 'dkVCard', 'diagnostics': []}

Normalising puts line endings, folding and case right — and leaves
`X-PHONE-MODEL`, which UniDAV has no opinion about, exactly where it was.
Normalising is not filtering.

In [3]:
unidav.normalize(card).split("\r\n")

['BEGIN:VCARD',
 'VERSION:4.0',
 'UID:urn:uuid:ada',
 'FN:Ada Lovelace',
 'EMAIL:ada@example.org',
 'X-PHONE-MODEL:something',
 'END:VCARD',
 '']

## The typed view

A host that wants a name and an address should not have to walk a component
tree. The projection is a *view*: take it to display, keep the document to
store.

In [4]:
projection = unidav.project(card)
projection["kind"], projection["uid"], projection["name"]["full"]

('contact', 'urn:uuid:ada', 'Ada Lovelace')

`x-phone-model` is absent from the projection and present in the
document. A host round-tripping through the projection alone would drop it.

In [5]:
"x-phone-model" in json.dumps(projection).lower()

False

## The JSON encodings

In [6]:
unidav.to_jcard(unidav.normalize(card))

['vcard',
 [['version', {}, 'text', '4.0'],
  ['uid', {}, 'uri', 'urn:uuid:ada'],
  ['fn', {}, 'text', 'Ada Lovelace'],
  ['email', {}, 'text', 'ada@example.org'],
  ['x-phone-model', {}, 'unknown', 'something']]]

## Merging is where a sync client lives

Two sides changed the same contact. `merge` takes the common ancestor and both
versions, and answers with a document plus the conflicts it could not decide —
rather than picking a winner quietly.

In [7]:
base  = "BEGIN:VCARD\r\nVERSION:4.0\r\nUID:a\r\nFN:Ada\r\nEND:VCARD\r\n"
local = base.replace("FN:Ada", "FN:Grace")

agreed = unidav.merge(base, local, base)
print("merged FN:Grace:", "FN:Grace" in agreed["document"])
print("conflicts:      ", agreed["conflicts"])

merged FN:Grace: True
conflicts:       []


In [8]:
both = base.replace("FN:Ada", "FN:Katherine")
clash = unidav.merge(base, local, both)
print("both sides changed it, conflicts:", len(clash["conflicts"]))

both sides changed it, conflicts: 1


## Recurrence

Every expansion is given a window and a ceiling. A rule with neither UNTIL nor
COUNT is unbounded by definition, so the caller says where to stop rather than
the library guessing.

In [9]:
unidav.expand_recurrence("20260105T090000Z", "FREQ=WEEKLY;BYDAY=MO",
                         "20260101T000000Z", "20260201T000000Z", 10)

['20260105T090000Z',
 '20260112T090000Z',
 '20260119T090000Z',
 '20260126T090000Z']

## The C ABI underneath

The same engine is reachable from anything that speaks C: strings in, strings
out.

```c
char *unidav_normalize(const char *input);
char *unidav_project_json(const char *input);
void  unidav_free(void *value);
```

There every returned string is the caller's, freed exactly once with
`unidav_free`, and a failure is a NULL return with a code in `unidav_status` —
an exception must never unwind across an ABI boundary.

See `include/UniDAV.h`, and the book for the full picture.